# Expression Pathway Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/02_expression_scoring.ipynb)

**What this does:** Scores biological pathways from bulk RNA-seq gene expression data using three methods (ssGSEA, GSVA, Mean-Z), then clusters patients into molecular subtypes.

**When to use:** You have a gene expression matrix (samples × genes) from RNA-seq or microarray, plus pathway definitions (GMT format).

**Prerequisites:** [00_quick_demo.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/00_quick_demo.ipynb)

In [ ]:
# Install pathway-subtyping
!pip install -q pathway-subtyping==0.3.0

import pathway_subtyping
print(f"pathway-subtyping v{pathway_subtyping.__version__}")

## 1. Generate Synthetic Expression Data

We'll use the built-in simulation module to create a realistic expression matrix with planted subtypes. In real use, you'd load your own data with `load_expression_matrix()`.

In [ ]:
from pathway_subtyping import (
    ExpressionSimulationConfig,
    generate_synthetic_expression_data,
    ExpressionInputType,
)

# Simulate 150 samples, 3 subtypes, 500 genes across 10 pathways
config = ExpressionSimulationConfig(
    n_samples=150,
    n_genes=500,
    n_pathways=10,
    n_genes_per_pathway=30,
    n_subtypes=3,
    effect_size=1.5,
    noise_level=1.0,
    seed=42,
    input_type=ExpressionInputType.TPM,
)

sim = generate_synthetic_expression_data(config)

print(f"Expression matrix: {sim.expression_matrix.shape[0]} samples × {sim.expression_matrix.shape[1]} genes")
print(f"Pathways: {len(sim.pathways)} ({', '.join(sim.pathway_names[:5])})...")
print(f"Planted subtypes: {len(set(sim.true_labels))} groups")
print(f"\nFirst 5 rows × 8 columns:")
sim.expression_matrix.iloc[:5, :8]

## 2. Score Pathways — Three Methods

The framework provides three scoring methods. Each converts a genes × samples expression matrix into a pathways × samples score matrix:

| Method | Algorithm | Speed | Best For |
|--------|-----------|-------|----------|
| **ssGSEA** | Rank-based enrichment | Medium | Recommended default |
| **GSVA** | Empirical CDF + KS statistic | Medium | Alternative to ssGSEA |
| **Mean-Z** | Z-score averaging | Fast | Quick exploration |

In [ ]:
from pathway_subtyping import (
    score_pathways_from_expression,
    ExpressionScoringMethod,
)

# Method 1: ssGSEA (recommended)
ssgsea_result = score_pathways_from_expression(
    sim.expression_matrix,
    sim.pathways,
    method=ExpressionScoringMethod.SSGSEA,
    seed=42,
)

print("ssGSEA Scoring")
print(f"  Output shape: {ssgsea_result.pathway_scores.shape}")
print(f"  Method: {ssgsea_result.method.value}")
print(f"  Pathways scored: {list(ssgsea_result.pathway_scores.columns[:5])}")
print(f"\nFirst 5 samples:")
ssgsea_result.pathway_scores.head()

In [ ]:
# Method 2: GSVA
gsva_result = score_pathways_from_expression(
    sim.expression_matrix,
    sim.pathways,
    method=ExpressionScoringMethod.GSVA,
    seed=42,
)

# Method 3: Mean-Z (fastest)
meanz_result = score_pathways_from_expression(
    sim.expression_matrix,
    sim.pathways,
    method=ExpressionScoringMethod.MEAN_Z,
    seed=42,
)

print("Method comparison (pathway scores correlation):")
import numpy as np

ssgsea_flat = ssgsea_result.pathway_scores.values.flatten()
gsva_flat = gsva_result.pathway_scores.values.flatten()
meanz_flat = meanz_result.pathway_scores.values.flatten()

print(f"  ssGSEA vs GSVA:   r = {np.corrcoef(ssgsea_flat, gsva_flat)[0,1]:.3f}")
print(f"  ssGSEA vs Mean-Z: r = {np.corrcoef(ssgsea_flat, meanz_flat)[0,1]:.3f}")
print(f"  GSVA vs Mean-Z:   r = {np.corrcoef(gsva_flat, meanz_flat)[0,1]:.3f}")

## 3. Cluster and Validate

Use the pathway scores to discover molecular subtypes. We'll use GMM clustering and validate with built-in gates.

In [ ]:
from pathway_subtyping import (
    run_clustering,
    ClusteringAlgorithm,
    ValidationGates,
)
from sklearn.metrics import adjusted_rand_score

# Cluster using ssGSEA scores
clustering = run_clustering(
    ssgsea_result.pathway_scores.values,
    n_clusters=3,
    algorithm=ClusteringAlgorithm.GMM,
    seed=42,
)

ari = adjusted_rand_score(sim.true_labels, clustering.labels)

print("Clustering Results (ssGSEA scores)")
print(f"  Silhouette:        {clustering.silhouette:.3f}")
print(f"  Calinski-Harabasz: {clustering.calinski_harabasz:.1f}")
print(f"  ARI vs truth:      {ari:.3f}")

# Validate
gates = ValidationGates(seed=42, n_permutations=50, n_bootstrap=50)

# Build a gene_burdens-like DataFrame from expression (use raw expression as proxy)
val = gates.run_all(
    pathway_scores=ssgsea_result.pathway_scores,
    cluster_labels=clustering.labels,
    pathways=sim.pathways,
    gene_burdens=sim.expression_matrix,
    n_clusters=3,
    gmm_seed=42,
)

print(f"\nValidation Gates")
for test in val.results:
    icon = "PASS" if test.passed else "FAIL"
    print(f"  [{icon}] {test.name}: {test.metric_name}={test.metric_value:.3f}")
print(f"  Overall: {'ALL PASSED' if val.all_passed else 'SOME FAILED'}")

## 4. Compare Methods Side-by-Side

Let's see which scoring method best recovers the planted subtypes.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

methods = {
    "ssGSEA": ssgsea_result.pathway_scores,
    "GSVA": gsva_result.pathway_scores,
    "Mean-Z": meanz_result.pathway_scores,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, scores) in zip(axes, methods.items()):
    # Cluster each
    cl = run_clustering(scores.values, n_clusters=3, algorithm=ClusteringAlgorithm.GMM, seed=42)
    ari = adjusted_rand_score(sim.true_labels, cl.labels)

    # PCA projection
    pca = PCA(n_components=2, random_state=42)
    X = pca.fit_transform(scores.values)

    for label in sorted(set(cl.labels)):
        mask = cl.labels == label
        ax.scatter(X[mask, 0], X[mask, 1], label=f"Cluster {label}", s=30, alpha=0.7)

    ax.set_title(f"{name}\nARI={ari:.3f}, Silhouette={cl.silhouette:.3f}")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
    ax.legend(fontsize=8)

plt.suptitle("Expression Scoring Methods Compared", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Loading Your Own Data

Replace the synthetic data with your own expression matrix:

```python
from pathway_subtyping import load_expression_matrix, ExpressionInputType

# Load from CSV (samples × genes, first column = sample IDs)
expr_matrix, qc_report = load_expression_matrix(
    "your_expression.csv",
    input_type=ExpressionInputType.TPM,  # or COUNTS, FPKM, LOG2
)
print(qc_report.format_report())

# Load pathways from GMT file
from pathway_subtyping.pipeline import load_gmt
pathways = load_gmt("c2.cp.reactome.v2023.2.Hs.symbols.gmt")

# Score and cluster
result = score_pathways_from_expression(expr_matrix, pathways, method=ExpressionScoringMethod.SSGSEA, seed=42)
clustering = run_clustering(result.pathway_scores.values, n_clusters=3, seed=42)
```

## Next Steps

- **Multi-omic fusion:** [03_multi_omic_fusion.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/03_multi_omic_fusion.ipynb) — combine expression + VCF scores
- **Deconvolution:** [04_deconvolution.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/04_deconvolution.ipynb) — estimate cell-type proportions from bulk
- **Visualization:** [05_visualization.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/05_visualization.ipynb) — interactive Plotly reports
- **API reference:** [Expression API](https://github.com/topmist-admin/pathway-subtyping-framework/blob/main/docs/api/expression.md)

---
*Built with [pathway-subtyping](https://pypi.org/project/pathway-subtyping/). Disease-agnostic. Open source.*